# Realized Volatility and the Sampling Interval

Realized volatility, the square root of the sum of squared intraday returns, measures how much a price varied over a day without a model of volatility ([Andersen and Bollerslev (1998)](https://doi.org/10.2307/2527343)). This tutorial simulates a trading day whose true volatility is known, measures it from the ticks with `simple_volatility` of HighFrequencyCovariance.jl ([Baumann and Klymak (2022)](https://doi.org/10.18637/jss.v103.i14)), and shows why the sampling interval matters when prices carry microstructure noise ([Aït-Sahalia, Mykland and Zhang (2005)](https://doi.org/10.1093/rfs/hhi016)).

In [ ]:
Pollis.packages("HighFrequencyCovariance", "Plots")

## A Simulated Trading Day

The efficient log price is a random walk with a daily volatility of 1%, observed every second for 6.5 hours. Each observed price adds independent noise of 2 basis points, as bid-ask bounce would. The ticks are a Tables.jl table, a named tuple of columns, and the one line that builds `ts` turns them into the `SortedDataFrame` the package works on. `grid_returns` samples the last price at or before each multiple of a number of seconds.

In [ ]:
using HighFrequencyCovariance, Dates, Random, Statistics
import Plots

rng = Xoshiro(7)
n = 23400  # a tick every second over a trading day of 6.5 hours
efficient = log(100) .+ cumsum(0.01 / sqrt(n) .* randn(rng, n))  # efficient log price, daily volatility 1%
noise = 0.0002 .* randn(rng, n)  # microstructure noise of 2 basis points
ticks = (time = collect(1.0:n), asset = fill(:SPX, n), logprice = efficient .+ noise)
ts = SortedDataFrame(HighFrequencyCovariance.DataFrame(ticks), :time, :asset, :logprice, Second(1))
span = duration(ts; in_dates_period = false)  # seconds from the first tick to the last

function grid_returns(t, p, every)  # log returns between the last prices at or before each multiple of every seconds
	grid = range(first(t), last(t); step = every)
	return diff([p[searchsortedlast(t, g)] for g in grid])
end

In [ ]:
Plots.plot(ticks.time ./ 3600, exp.(ticks.logprice); xlabel = "hours", ylabel = "price", label = "observed", title = "One simulated trading day")

## One Daily Return Against Many Intraday Returns

The absolute return of the whole day is a single draw, with an expected value of about 0.8 times the volatility and a wide spread around it. Realized volatility from five-minute returns adds the squares of about 78 returns and lands much closer to the true 1%.

In [ ]:
five_minutes = simple_volatility(ts, [:SPX]; fixed_spacing = 300)[:SPX] * sqrt(span)
(absolute_daily_return = abs(ticks.logprice[end] - ticks.logprice[1]), realized_five_minutes = five_minutes, true_volatility = 0.01)

## Realized Volatility on Every Tick

Using every tick should be better still, but every tick return carries the noise of two prices. Each squared return then adds about twice the noise variance, so realized variance gains roughly 2 n times the noise variance over n returns: far more than the true variance here.

In [ ]:
every_tick = simple_volatility(ts, [:SPX]; use_all_obs = true)[:SPX] * sqrt(span)
bias = 2 * (n - 1) * 0.0002^2  # expected noise in realized variance on every tick
(realized_every_tick = every_tick, predicted_from_noise = sqrt(0.01^2 + bias), true_volatility = 0.01)

## The Volatility Signature Plot

Realized volatility computed at intervals from one second to half an hour shows where the noise stops dominating. The curve falls steeply over the first few seconds and flattens near the true volatility once the interval is long enough for the change of the efficient price to outweigh the noise.

In [ ]:
intervals = [1, 2, 5, 10, 20, 30, 60, 120, 300, 600, 900, 1800]
signature = [sqrt(sum(grid_returns(ticks.time, ticks.logprice, k) .^ 2)) for k in intervals]
Plots.plot(intervals, 100 .* signature; xscale = :log10, marker = :circle, xlabel = "sampling interval (seconds)", ylabel = "realized volatility (%)", label = "realized volatility", title = "Volatility signature plot")
Plots.hline!([1.0]; linestyle = :dash, label = "true volatility")

## Fewer Returns Mean More Sampling Error

Long intervals remove the noise but leave few returns: half an hour gives only about a dozen in a day, and the estimate then varies more around the truth. The table shows the number of returns and the estimate at a few intervals.

In [ ]:
[(seconds = k, returns = length(grid_returns(ticks.time, ticks.logprice, k)), volatility = round(sqrt(sum(grid_returns(ticks.time, ticks.logprice, k) .^ 2)), sigdigits = 4)) for k in [60, 300, 900, 1800]]

## The Package's Own Choice of Interval

Called with no `fixed_spacing`, `simple_volatility` chooses a spacing itself from its estimate of the noise. Its answer can be read against the signature plot above.

In [ ]:
(package_choice = simple_volatility(ts, [:SPX])[:SPX] * sqrt(span), five_minutes = five_minutes, true_volatility = 0.01)

## Things to Try

- Set the noise to zero in the setup cell: the signature plot becomes flat and every tick gives the best estimate.
- Raise the noise to 10 basis points: the curve stays high for longer, and five minutes may no longer be enough.
- Compare with the **Two-Scales** tab of High-Frequency Volatility, which uses every tick and corrects for the noise.